# SmartPantry AI: Recall Monitoring and Safety Gate

**Production module:** `modules/recall_checker.py`  
**Automated tests:** `tests/test_recall_checker.py`  
**Development stage:** Phase 4 integration and validation  

## 1. Introduction and objectives

This notebook preserves the original Phase 3 FDA recall research and prototype experiments, then validates the current production recall-checker module with controlled tests. It covers FDA retrieval and pagination, USDA FSIS retrieval through a third-party service, normalization, matching by product name/brand/UPC/lot, and conservative recipe eligibility decisions.

**Safety interpretation:** `NO_MATCH` is not a verified declaration of safety. Both retrieval functions currently report `check_status="INCOMPLETE"` even when records are returned. A `POSSIBLE_MATCH` is not a confirmed recall; incomplete checks are blocked by default, with limited acknowledgment support for nonmatching results. The notebook does not establish that a product is recall-free.

## 2. Phase 3 research and historical development

The material in this section comes from the original Phase 3 notebook and is retained as **historical evidence**, not as the current production implementation. Prototype `CLEAR`/`RECALLED` labels, old FDA key usage, and USDA access conclusions must not be used as current validation claims. Historical code is shown for review but is not executed when the notebook is run from top to bottom.

# **Project Overview**

SmartPantry AI is a food management application designed to help users track pantry items, reduce food waste, and improve food safety. The project combines recall monitoring, pantry management, and recipe recommendations into a single workflow. A key feature of SmartPantry is its ability to identify potentially recalled food products before they are used in meal recommendations.

The recall-monitoring component retrieves recall information from official sources and evaluates pantry items using product names, brands, and UPC codes. Results are classified as **CLEAR**, **POSSIBLE_MATCH**, or **RECALLED** to help users make informed decisions about the food they consume. As the project progresses, additional features and integrations will continue to improve accuracy, usability, and overall safety.

# **Phase 1 – Research & Planning**

Defined the SmartPantry concept and project goals.
Researched pantry management and food recall systems.
Identified recall monitoring as a key project feature.

# **Phase 2 – Design & Development Planning**

Designed the recall-monitoring workflow.
Defined recall classifications: CLEAR, POSSIBLE_MATCH, and RECALLED.
Planned integration with pantry and recipe modules.

# ***2. FDA Recall Data Review***
Available FDA Recall Fields
The FDA openFDA Food Enforcement API provides recall information that can support product matching and recall investigation.

Fields examined during testing:

recall_number
recalling_firm
product_description
reason_for_recall
recall_initiation_date
status
distribution_pattern

# ***API Limitations***
The FDA API provides useful recall information but does not consistently provide all product identifiers.

Observed limitations include:

UPC values may be unavailable.
Lot numbers may be unavailable.
Product naming may differ from pantry records.
Manufacturer naming may vary.
Search results do not automatically confirm a product-specific recall.

# ***3. Official Recall Examples***
The following examples were collected from FDA recall search results and reviewed to evaluate the availability of product information, matching confidence, and potential limitations.

# ***Recall Example Review***



**Example-Product	Available-Information	Matching Confidence**


1	ELITE Milk Chocolate Coated Rice Cakes-Product, manufacturer, recall reason-Possible Match

2	Coffee Toffee-Product, manufacturer, recall reason-Possible Match

3	Fudge Brownie Cookie-Product, manufacturer, recall reason-Possible Match

4	Fish Family Farm Dairy Products-Product, manufacturer, recall reason-Possible Match

5	Poco Dolce Bittersweet Chocolate-Product, manufacturer, recall reason-Possible Match

> **Historical Phase 3 context:** The following reflects the prototype at that time, not the current production behavior.

# ***Recall Matching Scenarios***

**Confirmed Match**

Scenario:



*   UPC available
*  UPC matches recalled product



Result:



*   RECALLED


Reason:



*   UPC is treated as the highest-confidence identifier.


**Possible Match**

Scenario:



*   Product name and/or brand matches
*   UPC or lot information unavailable


Result:



*   POSSIBLE_MATCH


Reason:



*   Additional verification is required before confirming a recall.


**Incomplete Check**

Scenario:



*   Product appears similar to a recalled item
*   Required identifiers are unavailable
*   API source unavailable or incomplete

Result:


*   Unable to confirm recall status


Reason:



*   Missing recall data must not be treated as a confirmed match or verified clearance.



# ***4. FDA Data Retrieval***

The FDA openFDA Food Enforcement API was used as the primary recall source during development and testing.

The following section demonstrates retrieval of recall records and validates API connectivity.



**Historical prototype code (not executed):**

```python
from google.colab import userdata
import pandas as pd
import requests

# Read the key from Colab Secrets; the key itself is not shown.
try:
    api_key = userdata.get("OPENFDA_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Colab couldn't find OPENFDA_API_KEY. Check the Secrets panel "
        "and make sure Notebook access is enabled."
    ) from exc

if not api_key:
    raise RuntimeError(
        "The OPENFDA_API_KEY secret is empty. Add your key in the Secrets panel."
    )

# Ask openFDA for up to 5 food-recall records mentioning "milk".
url = "https://api.fda.gov/food/enforcement.json"
params = {
    "api_key": api_key,
    "search": "product_description:milk",
    "limit": 5,
}

response = requests.get(url, params=params, timeout=30)

if response.status_code == 200:
    results = response.json().get("results", [])
    recalls = pd.json_normalize(results)

    if recalls.empty:
        print("The request worked, but no matching records were returned.")
    else:
        columns_to_show = [
            "recall_number",
            "recalling_firm",
            "product_description",
            "reason_for_recall",
            "recall_initiation_date",
            "status",
        ]
        print(f"Success! Found {len(recalls)} matching record(s).")
        display(recalls.reindex(columns=columns_to_show).fillna(""))

elif response.status_code == 404:
    print("The request reached openFDA, but no records matched that search.")

elif response.status_code in (401, 403):
    print(
        "openFDA did not accept the request. Check that the API key was copied "
        "correctly into Colab Secrets and that the secret is named OPENFDA_API_KEY."
    )

else:
    print(f"The request returned HTTP status {response.status_code}.")
    print("Check your internet connection and try again.")

```

# **5. Historical Development Versions**

The following sections contain the original development experiments used during construction of the recall-monitoring prototype.

These cells are preserved for documentation purposes and demonstrate the progression from initial API retrieval to the final matching implementation.

**Historical prototype code (not executed):**

```python
pantry_item = "milk"

print(f"Checking recalls for: {pantry_item}")
```

**Historical prototype code (not executed):**

```python
matches = recalls[
    recalls["product_description"]
    .str.contains(pantry_item, case=False, na=False)
]

print(matches[[
    "product_description",
    "reason_for_recall",
    "status"
]])
```

**Historical prototype code (not executed):**

```python
if len(matches) > 0:
    print("RECALLED")
else:
    print("CLEAR")
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df):

    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    if len(matches) > 0:
        return "RECALLED"

    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
print(check_recall("milk", recalls))

print(check_recall("banana", recalls))
```

# ***6. Recall Matching Logic***

The recall-matching system was developed incrementally and evolved from simple product-name matching to support brand matching, UPC matching, and structured recall classifications.

The goal of the matching system is to classify items as:



*   CLEAR
*   POSSIBLE_MATCH
*   RECALLED


While preserving the distinction between search results and confirmed product-specific recalls.


**Historical prototype code (not executed):**

```python
import pandas as pd

def check_recall(product_name, recalls_df):

    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    if len(matches) > 0:
        return "RECALLED"

    return "CLEAR"


test_data = pd.DataFrame([
    {
        "product_description": "Milk Chocolate Rice Cakes"
    },
    {
        "product_description": "Frozen Pizza"
    }
])

print("Milk:", check_recall("milk", test_data))
print("Banana:", check_recall("banana", test_data))
```

**Historical prototype code (not executed):**

```python
import requests
import pandas as pd

def get_fda_recalls(api_key, search_term="milk", limit=5):
    url = "https://api.fda.gov/food/enforcement.json"
    params = {
        "api_key": api_key,
        "search": f"product_description:{search_term}",
        "limit": limit,
    }
    response = requests.get(url, params=params, timeout=30)
    if response.status_code == 200:
        results = response.json().get("results", [])
        return pd.json_normalize(results)
    return pd.DataFrame()
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df):
    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]
    if len(matches) > 0:
        return "RECALLED"
    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
import pandas as pd
import requests

def get_fda_recalls(api_key, search_term="milk", limit=5):

    url = "https://api.fda.gov/food/enforcement.json"

    params = {
        "api_key": api_key,
        "search": f"product_description:{search_term}",
        "limit": limit,
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        results = response.json().get("results", [])
        return pd.json_normalize(results)

    return pd.DataFrame()


def check_recall(product_name, recalls_df):

    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    if len(matches) > 0:
        return "RECALLED"

    return "CLEAR"


# TEST
recalls = get_fda_recalls(
    api_key,
    search_term="milk",
    limit=20
)

print("Records Found:", len(recalls))

print("Milk:", check_recall("milk", recalls))
print("Banana:", check_recall("banana", recalls))
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df):
    matches = recalls_df[
        recalls_df["product_description"]
        .str.contains(product_name, case=False, na=False)
    ]

    # Exact match
    for description in recalls_df["product_description"].fillna(""):
        if description.lower() == product_name.lower():
            return "RECALLED"

    # Partial match
    if len(matches) > 0:
        return "POSSIBLE_MATCH"

    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
print("Milk:", check_recall("milk", recalls))
print("Chocolate:", check_recall("chocolate", recalls))
print("Banana:", check_recall("banana", recalls))
print(recalls["product_description"].head())

```

**Historical prototype code (not executed):**

```python
recalls = get_fda_recalls(
    api_key,
    search_term="product",
    limit=50
)
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df):

    product_name = product_name.lower().strip()

    # Exact match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if description == product_name:
            return "RECALLED"

    # Partial match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
print("Milk:", check_recall("milk", recalls))
print("Chocolate:", check_recall("chocolate", recalls))
print("Banana:", check_recall("banana", recalls))
print("Laptop:", check_recall("laptop", recalls))
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df, brand=None):

    product_name = product_name.lower().strip()

    # Exact match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if description == product_name:
            return "RECALLED"

    # Brand + Product Match
    if brand:

        brand = brand.lower().strip()

        for description in recalls_df["product_description"].fillna(""):

            description = description.lower()

            if (
                brand in description
                and product_name in description
            ):
                return "POSSIBLE_MATCH"

    # Product-only partial match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
print(
    "Elite Chocolate:",
    check_recall(
        "chocolate",
        recalls,
        brand="elite"
    )
)

print(
    "Milk:",
    check_recall(
        "milk",
        recalls
    )
)

print(
    "Banana:",
    check_recall(
        "banana",
        recalls
    )
)
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df, brand=None):
    product_name = product_name.lower().strip()

    # Brand + Product Match
    if brand:
        brand = brand.lower().strip()
        for description in recalls_df["product_description"].fillna(""):
            description = description.lower()
            if brand in description and product_name in description:
                return "POSSIBLE_MATCH"

    # Product-only Match
    for description in recalls_df["product_description"].fillna(""):
        description = description.lower()
        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df, brand=None):
    # Brand + Product Match
    if brand:

        brand = brand.lower().strip()

        for description in recalls_df["product_description"].fillna(""):

            description = description.lower()

            if (
                brand in description
                and product_name in description
            ):
                return "POSSIBLE_MATCH"


    # Product-only Match
    for description in recalls_df["product_description"].fillna(""):

        description = description.lower()

        if product_name in description:
            return "POSSIBLE_MATCH"

    return "CLEAR"
```

**Historical prototype code (not executed):**

```python
print(
    "Elite Chocolate:",
    check_recall(
        "chocolate",
        recalls,
        brand="elite"
    )
)

print(
    "Milk:",
    check_recall(
        "milk",
        recalls
    )
)

print(
    "Banana:",
    check_recall(
        "banana",
        recalls
    )
)

print(
    "Laptop:",
    check_recall(
        "laptop",
        recalls
    )
)
```

**Historical prototype code (not executed):**

```python
{
    "status": "POSSIBLE_MATCH",
    "source": "FDA",
    "matched_product": "ELITE MILK CHOCOLATE COATED RICE CAKES"
}
```

**Historical prototype code (not executed):**

```python
"""
Returns a structured recall result.
"""
```

**Historical prototype code (not executed):**

```python
def check_exact_match(description, product_name):
    if description == product_name:
        return {
            "status": "RECALLED",
            "source": "FDA",
            "matched_product": description
        }
```

**Historical prototype code (not executed):**

```python
def check_partial_match(description):
    return {
        "status": "POSSIBLE_MATCH",
        "source": "FDA",
        "matched_product": description
    }
```

**Historical prototype code (not executed):**

```python
def check_match_result(description):
    return {
        "status": "POSSIBLE_MATCH",
        "source": "FDA",
        "matched_product": description
    }
```

**Historical prototype code (not executed):**

```python
def get_clear_status():
    return {
        "status": "CLEAR",
        "source": None,
        "matched_product": None
    }
```

**Historical prototype code (not executed):**

```python
result = check_recall(
    "chocolate",
    recalls,
    brand="elite"
)

print(result)
```

# ***7. Testing***

The following tests were created to validate recall classifications and matching confidence levels.

Test coverage includes:


*   UPC matching
*   Brand matching
*   Product matching
*   Clear/non-matching scenarios


**Historical prototype code (not executed):**

```python
test_recalls = pd.DataFrame([
    {
        "product_description": "Milk Chocolate Rice Cakes",
        "upc": "123456789"
    },
    {
        "product_description": "Frozen Pizza",
        "upc": "987654321"
    }
])
```

**Historical prototype code (not executed):**

```python
def check_recall(product_name, recalls_df, brand=None, upc=None):
    # 1. UPC Match (Highest Confidence)
    if upc:
        for _, row in recalls_df.iterrows():
            recall_upc = str(row.get("upc", "")).strip()
            if recall_upc and recall_upc == str(upc):
                return {
                    "status": "RECALLED",
                    "source": "FDA",
                    "matched_product": row["product_description"]
                }

    # Normalize input text
    product_name_clean = product_name.lower().strip()

    # 2. Exact Match
    for _, row in recalls_df.iterrows():
        desc = str(row.get("product_description", "")).lower().strip()
        if desc == product_name_clean:
            return {
                "status": "RECALLED",
                "source": "FDA",
                "matched_product": row["product_description"]
            }

    # 3. Brand + Product Match
    if brand:
        brand_clean = brand.lower().strip()
        for _, row in recalls_df.iterrows():
            desc = str(row.get("product_description", "")).lower()
            if brand_clean in desc and product_name_clean in desc:
                return {
                    "status": "POSSIBLE_MATCH",
                    "source": "FDA",
                    "matched_product": row["product_description"]
                }

    # 4. Product-Only Match
    for _, row in recalls_df.iterrows():
        desc = str(row.get("product_description", "")).lower()
        if product_name_clean in desc:
            return {
                "status": "POSSIBLE_MATCH",
                "source": "FDA",
                "matched_product": row["product_description"]
            }

    # 5. CLEAR
    return {
        "status": "CLEAR",
        "source": None,
        "matched_product": None
    }
```

**Historical prototype code (not executed):**

```python
result = check_recall(
    "Chocolate Rice Cakes",
    test_recalls,
    brand="elite",
    upc="123456789"
)

print(result)
```

**Historical prototype code (not executed):**

```python
result = check_recall(
    "Chocolate Rice Cakes",
    test_recalls,
    brand="elite",
    upc="555555555"
)

print(result)
```

**Historical prototype code (not executed):**

```python
import pandas as pd


def run_tests():

    recalls = pd.DataFrame([
        {
            "product_description": "Milk Chocolate Rice Cakes",
            "upc": "123456789"
        }
    ])

    print(
        "Test 1:",
        check_recall(
            "Chocolate Rice Cakes",
            recalls,
            upc="123456789"
        )
    )

    print(
        "Test 2:",
        check_recall(
            "chocolate",
            recalls,
            brand="elite"
        )
    )

    print(
        "Test 3:",
        check_recall(
            "banana",
            recalls
        )
    )


if __name__ == "__main__":
    run_tests()
```

**Historical prototype code (not executed):**

```python
import pandas as pd


def run_tests():

    recalls = pd.DataFrame([
        {
            "product_description": "Milk Chocolate Rice Cakes",
            "upc": "123456789"
        }
    ])

    # Test 1: UPC Match
    result = check_recall(
        "Chocolate Rice Cakes",
        recalls,
        upc="123456789"
    )

    assert result["status"] == "RECALLED"
    print("✅ Test 1 Passed: UPC Match")

    # Test 2: Brand + Product Match
    result = check_recall(
        "chocolate",
        recalls,
        brand="elite"
    )

    assert result["status"] == "POSSIBLE_MATCH"
    print("✅ Test 2 Passed: Brand Match")

    # Test 3: No Match
    result = check_recall(
        "banana",
        recalls
    )

    assert result["status"] == "CLEAR"
    print("✅ Test 3 Passed: Clear Result")


if __name__ == "__main__":
    run_tests()
```

**Historical prototype code (not executed):**

```python
    # Test 4: Unknown Product
    result = check_recall(
        "laptop",
        recalls
    )

    assert result["status"] == "CLEAR"
    print("✅ Test 4 Passed: Unknown Product")

    # Test 5: Product-Only Match
    result = check_recall(
        "milk",
        recalls
    )

    assert result["status"] == "POSSIBLE_MATCH"
    print("✅ Test 5 Passed: Product Match")
```

> **Historical Phase 3 context:** The following reflects the prototype at that time, not the current production behavior.

# ***8. Results***

The recall-monitoring prototype successfully demonstrated all three classification outcomes.

**Test-Expected Result-Actual Result-Status**

UPC Match-RECALLED-RECALLED-PASS

Brand Match-POSSIBLE_MATCH-POSSIBLE_MATCH-PASS

Product Match-POSSIBLE_MATCH-POSSIBLE_MATCH-PASS

Non-Match-CLEAR-CLEAR-PASS

Unknown Product-CLEAR-CLEAR-PASS

## 9. USDA FSIS: Phase 3 investigation and Phase 4 implementation

### Phase 3 historical finding

During Phase 3, the team investigated USDA FSIS recall data as a second source. The access method tested at that time returned HTTP 403 errors, so **the Phase 3 prototype relied on the FDA openFDA Food Enforcement API**. This was a limitation of the earlier prototype, **not the current implementation**.

### Phase 4 implementation: USDA FSIS is now integrated

The current `modules/recall_checker.py` includes `get_usda_recalls()`, which requests USDA FSIS recall information through the **third-party** endpoint `https://foodsafetydata-api.onrender.com/api/recalls`. It also includes `normalize_usda_recall()` to convert returned entries into the SmartPantry record format. The normalization captures source, recall ID, description, brand, classification, reason, and source URL; fields such as UPC and lot codes may be unavailable.

The current automated tests cover a simulated successful USDA response, a connection failure, an unexpected response, record normalization, and matching a normalized USDA record to a pantry product. These are **mocked tests**, not evidence that the live third-party service is currently available.

**Safety limitation:** `get_usda_recalls()` returns `check_status="INCOMPLETE"` even on a successful retrieval. Therefore, obtaining records does not establish comprehensive recall clearance. The FDA and USDA retrieval functions are available independently; this module does not automatically combine both sources into a complete verified check.


> **Historical Phase 3 context:** The following reflects the prototype at that time, not the current production behavior.

# ***10. Limitations***
The recall-monitoring prototype was designed to prioritize safety, traceability, and reproducibility. As a result, several limitations should be considered when interpreting recall results.



*   A missing recall match does not guarantee that a product is safe.
*   Missing UPC, lot number, or product information may reduce matching accuracy.
*   Product names may vary between pantry records and official recall listings.
*   If a recall source is unavailable or a connection fails, the result must not be treated as a successful recall check or verified clearance.
*   POSSIBLE_MATCH classifications require additional verification and should not be treated as confirmed recalls.
*   FDA recall data is limited to the information provided within available recall records.
*   SmartPantry currently relies on FDA recall data only. Products outside available FDA recall coverage may not be represented in search results.


These limitations reinforce the importance of treating recall monitoring as a decision-support tool rather than a guarantee of product safety.

> **Historical Phase 3 context:** The following reflects the prototype at that time, not the current production behavior.

# ***11. Integration Summary***
The recall-monitoring module serves as a safety gate within SmartPantry and uses FDA recall data as its primary verification source.

Pantry items can be evaluated using available product information, including product names, brands, and UPC values. Based on matching results, items are classified as:


*   CLEAR
*   POSSIBLE_MATCH
*   RECALLED




These classifications support downstream decision making before items continue to recipe recommendation workflows.

Items classified as RECALLED can be flagged for removal or excluded from recommendations. POSSIBLE_MATCH results may require additional user review and verification before proceeding. CLEAR results indicate that no matching FDA recall record was identified during the check.

The recall-monitoring component was designed to provide traceable, source-backed recall evaluations while remaining independent from AI-generated recommendations and conversational interactions.

> **Historical Phase 3 context:** The following reflects the prototype at that time, not the current production behavior.

# ***12. Final Validation***
The notebook was reviewed and executed to verify that recall retrieval, matching logic, and testing procedures functioned as expected within the final FDA-based recall-monitoring workflow.

Validation confirmed successful demonstration of:



*   FDA recall data retrieval
*   Brand-aware matching
*   UPC-based matching
*   CLEAR classifications
*   POSSIBLE_MATCH classifications
*   RECALLED classifications
*   Product-name matching

The notebook also confirmed that simulated test cases can be executed without requiring an FDA API key, supporting reproducibility and easier evaluation.

All validation tests produced expected outcomes and demonstrated the intended behavior of the recall-monitoring prototype. Results were reviewed to ensure that recall classifications remained traceable to available product information and did not treat missing data or unavailable sources as successful recall checks.

The final notebook provides a reproducible record of development, testing, and validation activities completed for the SmartPantry recall-monitoring component.

## Phase 3 recall implementation summary (historical)

**Completed during Phase 3:**

- Connected to the FDA openFDA API and retrieved recall records.
- Prototyped product-name, brand, and UPC matching.
- Produced structured recall results and performed prototype validation.

**Not completed during Phase 3:** USDA FSIS integration and full application integration.

**Phase 4 update:** USDA FSIS retrieval and normalization **have since been implemented** in the production module through a third-party API. Current matching and recipe-eligibility functions have also been expanded. See Sections 3–8 for the production implementation and validation.


## Future improvements: updated for Phase 4

USDA FSIS integration is **no longer a future-only feature**: the production module now retrieves and normalizes USDA FSIS records using a third-party API. Further work should focus on improving the reliability and verification of that integration rather than treating it as unimplemented.

Remaining improvements include:

- Evaluate the availability, provenance, freshness, and long-term reliability of the third-party USDA FSIS feed; consider a supported authoritative data source when feasible.
- Define and validate a complete multi-source recall-check workflow. Both retrieval functions currently report `INCOMPLETE`, including after successful retrieval.
- Improve matching accuracy and product-name normalization, including generic ingredients, brands, UPCs, and lot identifiers.
- Extend real-world validation and end-to-end integration with pantry inventory and recipe recommendations.
- Maintain conservative safety decisions when recall records are missing, incomplete, or ambiguous.

These are **remaining improvements**, not a list of features absent from the current module.


> **Historical Phase 3 context:** The following reflects the prototype at that time, not the current production behavior.

# **Reflection**

The recall-monitoring component evolved from a research concept into a functioning prototype during the SmartPantry project. Through investigation of recall data sources, implementation of FDA recall retrieval, development of matching logic, and testing of recall classifications, the project established a practical foundation for food-safety monitoring.

One of the most important lessons learned was the value of separating safety-critical decisions from generative AI features. While AI can improve user interaction and recommendations, recall verification benefits from deterministic logic that can be traced back to official sources and consistently reproduced.

The development process also highlighted the importance of testing, documentation, and realistic system design decisions. Evaluating limitations, handling incomplete information, and documenting unavailable data sources helped ensure that recall results were communicated responsibly.

Overall, this work demonstrated how software engineering, data validation, and food-safety considerations can be combined to support a more informed pantry-management experience. The final prototype provides a strong foundation for future SmartPantry improvements and integration efforts.

## 3. Phase 4 production implementation: what changed

| Area | Historical prototype | Current production module |
|---|---|---|
| Recall sources | FDA, USDA investigated but inaccessible in original workflow | FDA retrieval plus USDA FSIS via a third-party API |
| FDA API | Notebook examples used a Colab secret and DataFrame | `get_fda_recalls(search_term, limit=None)` uses `requests` and pagination |
| USDA | Earlier access attempts returned HTTP 403 | `get_usda_recalls()` retrieves and normalizes third-party data; availability is not guaranteed |
| Product matching | Several evolving `check_recall` functions | `match_recall_to_product(product, recall)` |
| Matching status | Prototype used `CLEAR` and sometimes `RECALLED` directly | Individual match returns `POSSIBLE_MATCH` or `NO_MATCH`; aggregation can prioritize `RECALLED` |
| Retrieval completeness | Earlier experiments could treat no matches as clear | Retrieval results retain `check_status="INCOMPLETE"` |
| Recipe gate | Prototype classification examples | `is_recipe_eligible(result, acknowledge_incomplete=False)` |

**Important:** An exact UPC match in the current matching function produces `POSSIBLE_MATCH`, not an automatically confirmed `RECALLED` result. FDA/USDA records and the user's product identifiers still require appropriate verification. The USDA source is a third-party service, not a direct USDA-hosted API.

## 4. Google Colab setup and production imports

The setup below uses the SmartPantry repository and does not require a live FDA key. If the repository is already cloned, it reuses that checkout. For reliable results, verify that the checked-out branch contains the same module and test versions as those reviewed for this notebook.

In [1]:
from pathlib import Path
import os

repo = Path('/content/smartpantry-ai')
if not repo.is_dir():
    !git clone -b docs/phase4-notebook-updates https://github.com/SmartPantry-AI/smartpantry-ai.git /content/smartpantry-ai
%cd /content/smartpantry-ai
%pip install -q -r requirements.txt

from modules.recall_checker import (
    normalize_recall_record,
    normalize_usda_recall,
    get_fda_recalls,
    get_usda_recalls,
    match_recall_to_product,
    evaluate_recall_matches,
    is_recipe_eligible,
)

assert Path('modules/recall_checker.py').is_file()
print('PASS - Current recall checker module imported')

Cloning into '/content/smartpantry-ai'...
remote: Enumerating objects: 754, done.
remote: Counting objects: 100% (294/294), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 754 (delta 194), reused 178 (delta 128), pack-reused 460 (from 2)
Receiving objects: 100% (754/754), 16.82 MiB | 25.15 MiB/s, done.
Resolving deltas: 100% (344/344), done.
/content/smartpantry-ai
PASS - Current recall checker module imported


## 5. Controlled production behavior checks

These deterministic checks exercise production functions with simulated records. No network calls are made.

In [2]:
fda_record = normalize_recall_record({
    'source': 'FDA', 'recall_number': 'F-TEST-001',
    'product_description': 'Test Spinach',
})
assert fda_record['recall_id'] == 'F-TEST-001'
assert len(fda_record) == 9
assert fda_record['upc'] is None

usda_record = normalize_usda_recall({
    'recall_number': 'USDA-TEST-001', 'company': 'Example Foods',
    'title': 'Chicken Breast 16 oz', 'classification': 'Class I',
    'reason': 'Simulated contamination',
})
assert usda_record['source'] == 'USDA_FSIS'
assert usda_record['recall_id'] == 'USDA-TEST-001'
assert usda_record['recall_status'] == 'UNKNOWN'
print('PASS - FDA and USDA normalization')

PASS - FDA and USDA normalization


In [3]:
spinach = {'product_name': 'Baby Spinach', 'upc': '012345678905'}
recall = {
    'product_description': 'Fresh Spinach', 'brand': None,
    'upc': '012345678905', 'lot_codes': None,
}
assert match_recall_to_product(spinach, recall)['status'] == 'POSSIBLE_MATCH'
assert match_recall_to_product(
    {'product_name': 'Milk'},
    {'product_description': 'Fresh Baby Spinach 10 oz bags', 'brand': None,
     'upc': None, 'lot_codes': None}
)['status'] == 'NO_MATCH'
assert match_recall_to_product(
    {'product_name': 'Ham'},
    {'product_description': 'Frozen Hamburger Patties', 'brand': None,
     'upc': None, 'lot_codes': None}
)['status'] == 'NO_MATCH'
assert evaluate_recall_matches([
    {'status': 'NO_MATCH'}, {'status': 'POSSIBLE_MATCH'}
])['status'] == 'POSSIBLE_MATCH'
assert evaluate_recall_matches([
    {'status': 'RECALLED'}, {'status': 'POSSIBLE_MATCH'}
])['status'] == 'RECALLED'
print('PASS - Matching and safety-priority aggregation')

PASS - Matching and safety-priority aggregation


In [4]:
# Incomplete results are blocked by default.
assert not is_recipe_eligible({'status': 'NO_MATCH', 'check_status': 'INCOMPLETE'})
# Explicit acknowledgment can permit a nonmatching incomplete check,
# but it does not turn the result into verified clearance.
assert is_recipe_eligible(
    {'status': 'NO_MATCH', 'check_status': 'INCOMPLETE'},
    acknowledge_incomplete=True,
)
assert not is_recipe_eligible(
    {'status': 'POSSIBLE_MATCH', 'check_status': 'INCOMPLETE'},
    acknowledge_incomplete=True,
)
assert not is_recipe_eligible(
    {'status': 'RECALLED', 'check_status': 'COMPLETE'},
    acknowledge_incomplete=True,
)
assert not is_recipe_eligible({}, acknowledge_incomplete=True)
assert not is_recipe_eligible(evaluate_recall_matches([]))
print('PASS - Incomplete checks and recall safety gate')

PASS - Incomplete checks and recall safety gate


### USDA FSIS production validation (mocked, no network access)

This check exercises the **actual** `get_usda_recalls()` function with a simulated third-party response, then verifies normalization and product matching. It does not claim live service availability or complete recall clearance.


In [5]:
from unittest.mock import patch
from modules.recall_checker import get_usda_recalls

class SimulatedUSDAResponse:
    def raise_for_status(self):
        pass

    def json(self):
        return {
            "count": 1,
            "returned": 1,
            "recalls": [{
                "recall_number": "USDA-DEMO-001",
                "company": "Example Foods",
                "title": "Chicken Breast 16 oz",
                "classification": "Class I",
                "reason": "Simulated contamination",
                "url": "https://example.com/recall",
            }],
        }

with patch("modules.recall_checker.requests.get", return_value=SimulatedUSDAResponse()) as mocked_get:
    usda_result = get_usda_recalls()

assert mocked_get.call_count == 1
assert usda_result["error"] is None
assert usda_result["check_status"] == "INCOMPLETE"
assert len(usda_result["records"]) == 1
assert usda_result["records"][0]["source"] == "USDA_FSIS"
assert match_recall_to_product(
    {"product_name": "Chicken Breast", "brand": "Example Foods"},
    usda_result["records"][0],
)["status"] == "POSSIBLE_MATCH"
print("PASS - USDA retrieval, normalization, and matching (mocked)")


PASS - USDA retrieval, normalization, and matching (mocked)


## 6. Automated recall-checker test suite

Run the repository's dedicated `tests/test_recall_checker.py` tests. These tests use mocked FDA/USDA responses for retrieval scenarios and do **not** require live API calls. Coverage includes normalization, pagination and malformed responses, matching, aggregation, and acknowledgment behavior. The result below is generated at runtime; no pass count is asserted in advance.

In [6]:
import subprocess
import sys
from pathlib import Path

assert Path('tests/test_recall_checker.py').is_file(), 'Recall test file missing from checked-out branch'
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_recall_checker.py', '-v'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, 'Recall checker tests did not all pass'
print('PASS - Dedicated recall-checker automated tests')

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/smartpantry-ai
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 39 items

tests/test_recall_checker.py::test_normalize_recall_record PASSED        [  2%]
tests/test_recall_checker.py::test_normalize_fda_recall_number PASSED    [  5%]
tests/test_recall_checker.py::test_normalize_usda_standardized_record PASSED [  7%]
tests/test_recall_checker.py::test_incomplete_recall_check_blocks_recipe PASSED [ 10%]
tests/test_recall_checker.py::test_fda_retrieval_success PASSED          [ 12%]
tests/test_recall_checker.py::test_fda_retrieval_connection_error PASSED [ 15%]
tests/test_recall_checker.py::test_fda_retrieval_invalid_json PASSED     [ 17%]
tests/test_recall_checker.py::test_fda_retrieval_no_results PASSED       [ 20%]
tests/test_recal

## 7. Results and interpretation

**Historical Phase 3 results:** The earlier notebook reported prototype outcomes for UPC, brand, name, nonmatch, and unknown-product scenarios. Those results remain visible in the historical section, but their `CLEAR` and `RECALLED` labels do **not** represent current production decisions.

**Current Phase 4 validation:** Record actual results from Sections 5 and 6 after running the notebook. Passing controlled checks and mocked tests validates the specified code paths, not live coverage or real-world recall accuracy.

| Validation | Expected evidence |
|---|---|
| Production imports | Setup cell passes |
| FDA/USDA normalization | Controlled check passes |
| USDA FSIS retrieval and matching (mocked) | Controlled check passes |
| Matching and aggregation | Controlled check passes |
| Recipe safety gate | Controlled check passes |
| Dedicated pytest suite | Runtime pytest output (record count after execution) |

**No live FDA or USDA API request is performed by the new validation cells.**

## 8. Limitations and next steps

- FDA and third-party USDA retrieval currently return `INCOMPLETE` even after successful retrieval. A returned record list does not establish complete recall coverage.
- Matching is heuristic. Product descriptions, brands, UPCs, and lot identifiers may be missing or inconsistent. A `POSSIBLE_MATCH` requires verification, not an automatic conclusion that the item is recalled.
- USDA FSIS retrieval **is implemented**, but it depends on a third-party API that may be unavailable or change its response format. The live service was not verified by these mocked tests.
- The generic-ingredient safeguard currently treats some ingredient names specially; broader food-name normalization and coverage would require additional testing.
- The recall checker is a decision-support component, not a guarantee of food safety. End-to-end UI, account, pantry persistence, and live API behavior are outside the scope of these notebook tests.

**Next steps:** Run the notebook top to bottom in Colab, record the actual pytest count, and only then commit the updated notebook to `docs/phase4-notebook-updates`. Keep that branch open until the remaining notebooks are reviewed.